# 🧪 Notebook 02 — Data Preprocessing Experiments
## Context-Aware Neural Recommendation Engine
### H&M Personalized Fashion Recommendations

---

## 📌 Purpose of This Notebook

This is an **experimentation notebook** — a safe sandbox where we:
- Explore the raw data before touching production scripts
- Test preprocessing ideas and observe their effects
- Document our decisions with reasoning
- Identify problems **before** they break the pipeline

> **Why notebooks for experimentation?**  
> In real ML engineering teams, notebooks serve as a **laboratory**. You try things, break things, visualize intermediate results, and only once you're confident — you move that logic into clean, modular Python scripts (`src/`). This workflow is used at companies like Spotify, Netflix, and Zalando.

---

## 🗺️ How This Fits Into the ML Pipeline

```
RAW DATA  →  [THIS NOTEBOOK]  →  PRODUCTION SCRIPTS  →  FEATURES  →  MODEL  →  RECOMMENDATIONS
              (experiment)         (src/preprocessing)   (src/features)  (src/models)
```

**Preprocessing is the foundation.** Dirty data = broken model. No fancy neural network can fix bad inputs.

---

## 📂 Datasets
| File | Description |
|------|-------------|
| `customers.csv` | Customer profiles — demographics, membership status |
| `articles.csv` | Product catalog — clothing items with attributes |
| `transactions_train.csv` | Purchase history — who bought what and when |

---

## 📦 Section 1 — Import Libraries

We use only **pandas** and **numpy** at this stage.  
Why? Keeping dependencies minimal ensures this preprocessing logic is portable and reproducible.  
Advanced libraries (sklearn, etc.) come in the feature engineering phase.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')  # suppress harmless warnings during exploration

# Display settings — show more rows/columns so we can actually see the data
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Libraries loaded successfully")
print(f"   pandas  version: {pd.__version__}")
print(f"   numpy   version: {np.__version__}")

## 📂 Section 2 — Load Raw Datasets

We always load from the **raw/** folder and never modify those files directly.  
Raw data is sacred — treat it as read-only. All cleaning happens on copies.

In [ ]:
# ── Path Configuration ──────────────────────────────────────────────────────
# Using relative paths so this notebook works on any machine
RAW_DATA_PATH = '../data/raw/'
PROCESSED_PATH = '../data/processed/'
TEMP_PATH = '../data/sample/'  # for saving temporary experiment outputs

# Create output directories if they don't exist
os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(TEMP_PATH, exist_ok=True)

print("📁 Directory structure ready")

In [ ]:
# ── Load All Three Datasets ──────────────────────────────────────────────────
print("⏳ Loading datasets... (transactions may take a moment — it's large)")

customers_df = pd.read_csv(RAW_DATA_PATH + 'customers.csv')
print("✅ customers.csv loaded")

articles_df = pd.read_csv(RAW_DATA_PATH + 'articles.csv')
print("✅ articles.csv loaded")

transactions_df = pd.read_csv(RAW_DATA_PATH + 'transactions_train.csv')
print("✅ transactions_train.csv loaded")

print("\n🎉 All datasets loaded successfully!")

## 📐 Section 3 — Shape & Basic Overview

Before anything else, understand **how big** the data is.  
This tells us:
- Memory requirements
- Processing time expectations
- Scale of the recommendation problem

In [ ]:
# ── Dataset Shapes ───────────────────────────────────────────────────────────
print("=" * 55)
print("         DATASET SHAPES & MEMORY OVERVIEW")
print("=" * 55)

datasets = {
    'customers':    customers_df,
    'articles':     articles_df,
    'transactions': transactions_df
}

for name, df in datasets.items():
    rows, cols = df.shape
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"\n📄 {name.upper()}")
    print(f"   Rows    : {rows:,}")
    print(f"   Columns : {cols}")
    print(f"   Memory  : {mem_mb:.2f} MB")

print("\n" + "=" * 55)

In [ ]:
# ── Quick Peek at Each Dataset ───────────────────────────────────────────────
print("\n--- CUSTOMERS (first 3 rows) ---")
display(customers_df.head(3))

print("\n--- ARTICLES (first 3 rows) ---")
display(articles_df.head(3))

print("\n--- TRANSACTIONS (first 3 rows) ---")
display(transactions_df.head(3))

## 🔍 Section 4 — Missing Value Analysis

Missing values are **the #1 data quality issue** in real-world datasets.  
For a recommendation system, missing data in key fields like `customer_id` or `article_id` is critical — we cannot make recommendations without identifiers.

**Strategy overview:**
- **Identifier columns** (IDs): If missing → drop that row entirely
- **Categorical columns** (text categories): Fill with `'Unknown'` or most frequent value
- **Numerical columns**: Fill with median (robust to outliers) or 0 depending on context
- **High-missing columns** (>50% missing): Consider dropping the column entirely

In [ ]:
def analyze_missing_values(df, dataset_name):
    """
    Produces a clean missing value report for a dataframe.
    
    Shows: column name, missing count, missing percentage, and recommended action.
    Helps us decide HOW to handle each column's nulls before writing production code.
    """
    total_rows = len(df)
    missing_info = []
    
    for col in df.columns:
        missing_count = df[col].isnull().sum()
        missing_pct   = (missing_count / total_rows) * 100
        dtype         = str(df[col].dtype)
        
        # Suggest action based on missing % and dtype
        if missing_count == 0:
            action = '✅ No action needed'
        elif missing_pct > 70:
            action = '🗑️  Consider dropping column'
        elif missing_pct > 30:
            action = '⚠️  High missing — fill carefully'
        elif dtype == 'object':
            action = '🔤 Fill with Unknown / mode'
        elif 'float' in dtype or 'int' in dtype:
            action = '🔢 Fill with median or 0'
        else:
            action = '🔍 Investigate'
        
        missing_info.append({
            'Column':      col,
            'Dtype':       dtype,
            'Missing #':   missing_count,
            'Missing %':   round(missing_pct, 2),
            'Suggested Action': action
        })
    
    result_df = pd.DataFrame(missing_info)
    # Sort: worst missing first so we see problems immediately
    result_df = result_df.sort_values('Missing %', ascending=False)
    
    print(f"\n{'='*65}")
    print(f"  MISSING VALUE REPORT — {dataset_name.upper()}")
    print(f"  Total rows: {total_rows:,}  |  Total columns: {len(df.columns)}")
    print(f"{'='*65}")
    
    return result_df


# Run analysis on all three datasets
customers_missing    = analyze_missing_values(customers_df, 'customers')
articles_missing     = analyze_missing_values(articles_df, 'articles')
transactions_missing = analyze_missing_values(transactions_df, 'transactions')

In [ ]:
print("\n📋 CUSTOMERS — Missing Value Report")
display(customers_missing)

print("\n📋 ARTICLES — Missing Value Report")
display(articles_missing)

print("\n📋 TRANSACTIONS — Missing Value Report")
display(transactions_missing)

## 🔁 Section 5 — Duplicate Analysis

Duplicates can distort recommendation models:  
- A customer appearing twice inflates their interaction count  
- Duplicate transactions make products appear more popular than they are  
- This directly corrupts the collaborative filtering signal

**What counts as a duplicate depends on context:**
- `customers`: same `customer_id` = true duplicate
- `articles`: same `article_id` = true duplicate  
- `transactions`: same (`customer_id` + `article_id` + `t_dat`) = true duplicate purchase event

In [ ]:
def analyze_duplicates(df, dataset_name, key_columns=None):
    """
    Checks for duplicate rows.
    
    key_columns: if provided, checks duplicates based on specific ID columns
                 rather than the entire row — more meaningful for business logic.
    """
    total_rows = len(df)
    
    # Check full row duplicates
    full_dups = df.duplicated().sum()
    
    print(f"\n{'='*50}")
    print(f"  DUPLICATE REPORT — {dataset_name.upper()}")
    print(f"{'='*50}")
    print(f"  Total rows         : {total_rows:,}")
    print(f"  Full row duplicates: {full_dups:,} ({full_dups/total_rows*100:.3f}%)")
    
    # Check key-based duplicates if key columns are provided
    if key_columns:
        key_dups = df.duplicated(subset=key_columns).sum()
        print(f"  Key col duplicates : {key_dups:,} ({key_dups/total_rows*100:.3f}%)")
        print(f"  Key columns checked: {key_columns}")
    
    print(f"  ➜ Action: {'Remove duplicates' if full_dups > 0 else 'No action needed'}")


analyze_duplicates(customers_df,    'customers',    key_columns=['customer_id'])
analyze_duplicates(articles_df,     'articles',     key_columns=['article_id'])
analyze_duplicates(transactions_df, 'transactions', key_columns=['customer_id', 'article_id', 't_dat'])

## 🏷️ Section 6 — Data Type Checks

Correct data types are essential for:
- Memory efficiency (e.g., `int8` vs `int64`)
- Correct sorting and filtering (dates stored as strings break time-series logic)
- Model compatibility (ML models need numeric types)

**Common issues we look for:**
- Dates stored as `object` (string) instead of `datetime`
- IDs stored as `float` due to NaN contamination
- Categorical columns stored as `object` (fixable with `category` dtype → saves memory)

In [ ]:
def check_dtypes(df, dataset_name):
    """
    Displays column data types and flags potential issues.
    """
    dtype_info = []
    
    for col in df.columns:
        dtype       = str(df[col].dtype)
        sample_val  = df[col].dropna().iloc[0] if df[col].notna().any() else 'N/A'
        unique_vals = df[col].nunique()
        
        # Flag potential type issues
        flag = ''
        if col.endswith('_id') and dtype in ['float64', 'object']:
            flag = '⚠️  ID should be string/int'
        elif 'date' in col.lower() or col in ['t_dat'] and dtype == 'object':
            flag = '📅 Convert to datetime'
        elif dtype == 'object' and unique_vals < 50:
            flag = '🏷️  Low cardinality — consider category dtype'
        elif dtype == 'float64' and df[col].dropna().apply(lambda x: x == int(x)).all():
            flag = '🔢 Might be integer with NaNs'
        else:
            flag = '✅ OK'
        
        dtype_info.append({
            'Column':       col,
            'Current Type': dtype,
            'Sample Value': str(sample_val)[:30],
            'Unique Values': unique_vals,
            'Flag':         flag
        })
    
    result = pd.DataFrame(dtype_info)
    print(f"\n{'='*65}")
    print(f"  DATA TYPE REPORT — {dataset_name.upper()}")
    print(f"{'='*65}")
    return result


customers_types    = check_dtypes(customers_df,    'customers')
articles_types     = check_dtypes(articles_df,     'articles')
transactions_types = check_dtypes(transactions_df, 'transactions')

print("\n📋 CUSTOMERS — Data Types")
display(customers_types)

print("\n📋 ARTICLES — Data Types")
display(articles_types)

print("\n📋 TRANSACTIONS — Data Types")
display(transactions_types)

## 📅 Section 7 — Date Conversion Experiment

The `t_dat` column in transactions is a date — but likely stored as a string.  
We need it as `datetime` so we can:
- Sort transactions chronologically
- Calculate recency (days since last purchase)
- Create time-based features (month, week, season)
- Split train/test by time (avoiding data leakage)

> **Key concept:** In recommendation systems, time matters enormously.  
> A purchase from 3 years ago is far less relevant than one from last week.

In [ ]:
# ── Test date conversion on a copy ───────────────────────────────────────────
# ALWAYS experiment on a copy — never mutate the original during exploration
transactions_exp = transactions_df.copy()

print(f"Before conversion: dtype = {transactions_exp['t_dat'].dtype}")
print(f"Sample value     : {transactions_exp['t_dat'].iloc[0]}")

# Convert string date to datetime
transactions_exp['t_dat'] = pd.to_datetime(transactions_exp['t_dat'])

print(f"\nAfter conversion : dtype = {transactions_exp['t_dat'].dtype}")
print(f"Sample value     : {transactions_exp['t_dat'].iloc[0]}")
print(f"Date range       : {transactions_exp['t_dat'].min()} → {transactions_exp['t_dat'].max()}")

# Derive useful time features — these will be critical for the ML model
transactions_exp['year']       = transactions_exp['t_dat'].dt.year
transactions_exp['month']      = transactions_exp['t_dat'].dt.month
transactions_exp['week']       = transactions_exp['t_dat'].dt.isocalendar().week.astype(int)
transactions_exp['day_of_week']= transactions_exp['t_dat'].dt.dayofweek  # 0=Monday

print("\n✅ Derived time features:")
print(transactions_exp[['t_dat', 'year', 'month', 'week', 'day_of_week']].head(5))

## 🛠️ Section 8 — Null Handling Strategy (Decisions)

This is a **decision section** — we document our null handling choices here before coding them into production scripts.  
Good teams document WHY they made each choice, not just what they did.

### Strategy Summary

| Dataset | Column | Strategy | Reason |
|---------|--------|----------|---------|
| customers | `FN`, `Active` | Fill with `0` | Binary flag — null likely means inactive/no response |
| customers | `club_member_status` | Fill `'Unknown'` | Categorical — preserve record, unknown is valid state |
| customers | `fashion_news_frequency` | Fill `'None'` | Categorical — null = not subscribed |
| customers | `age` | Fill with **median** | Numerical — median is robust to outliers |
| articles | text/category columns | Fill `'Unknown'` | Categorical — unknown department/category is valid |
| articles | `detail_desc` | Fill `'No description'` | Text field — prevent downstream NLP errors |
| transactions | any null | Drop row | Transactions MUST have customer + article + date |

In [ ]:
# ── Experiment: Null Handling on Customers ───────────────────────────────────
customers_clean = customers_df.copy()

# 1. Binary flags — nulls likely mean 0 (not flagged)
for col in ['FN', 'Active']:
    if col in customers_clean.columns:
        before = customers_clean[col].isnull().sum()
        customers_clean[col] = customers_clean[col].fillna(0).astype(int)
        print(f"✅ '{col}': filled {before:,} nulls with 0")

# 2. Categorical text columns
cat_fill_map = {
    'club_member_status':      'Unknown',
    'fashion_news_frequency':  'None',
}
for col, fill_val in cat_fill_map.items():
    if col in customers_clean.columns:
        before = customers_clean[col].isnull().sum()
        customers_clean[col] = customers_clean[col].fillna(fill_val)
        print(f"✅ '{col}': filled {before:,} nulls with '{fill_val}'")

# 3. Age — fill with median (median is safer than mean for skewed data)
if 'age' in customers_clean.columns:
    age_median = customers_clean['age'].median()
    before = customers_clean['age'].isnull().sum()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print(f"✅ 'age': filled {before:,} nulls with median = {age_median:.1f}")

# Verify no nulls remain
remaining_nulls = customers_clean.isnull().sum().sum()
print(f"\n📊 Remaining nulls in customers: {remaining_nulls}")

In [ ]:
# ── Experiment: Null Handling on Articles ────────────────────────────────────
articles_clean = articles_df.copy()

# Identify all object (text) columns and fill nulls
object_cols = articles_clean.select_dtypes(include='object').columns.tolist()

for col in object_cols:
    null_count = articles_clean[col].isnull().sum()
    if null_count > 0:
        fill_val = 'No description' if 'desc' in col.lower() else 'Unknown'
        articles_clean[col] = articles_clean[col].fillna(fill_val)
        print(f"✅ '{col}': filled {null_count:,} nulls with '{fill_val}'")

# Numerical columns — fill with median
num_cols = articles_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    null_count = articles_clean[col].isnull().sum()
    if null_count > 0:
        median_val = articles_clean[col].median()
        articles_clean[col] = articles_clean[col].fillna(median_val)
        print(f"✅ '{col}': filled {null_count:,} nulls with median = {median_val}")

remaining_nulls = articles_clean.isnull().sum().sum()
print(f"\n📊 Remaining nulls in articles: {remaining_nulls}")

In [ ]:
# ── Experiment: Null Handling on Transactions ────────────────────────────────
transactions_clean = transactions_df.copy()

before_rows = len(transactions_clean)

# Transactions are events — partial records are useless for collaborative filtering
# A transaction without customer_id or article_id cannot be used
critical_cols = ['customer_id', 'article_id', 't_dat']
transactions_clean = transactions_clean.dropna(subset=critical_cols)

after_rows = len(transactions_clean)
dropped = before_rows - after_rows

print(f"Rows before: {before_rows:,}")
print(f"Rows after : {after_rows:,}")
print(f"Dropped    : {dropped:,} rows ({dropped/before_rows*100:.4f}%)")
print(f"\n📊 Remaining nulls in transactions: {transactions_clean.isnull().sum().sum()}")

## 🧹 Section 9 — Initial Cleaning Experiments

Now we perform the core cleaning steps and observe the results.  
Each step is isolated so we can see exactly what changed and why.

In [ ]:
# ── CUSTOMERS: Full Cleaning Experiment ──────────────────────────────────────
customers_clean = customers_df.copy()

# Step 1: Remove duplicate customer records
before = len(customers_clean)
customers_clean = customers_clean.drop_duplicates(subset=['customer_id'])
print(f"[1] Duplicates removed : {before - len(customers_clean):,} rows")

# Step 2: Drop rows with no customer_id (cannot be used in any recommendation)
before = len(customers_clean)
customers_clean = customers_clean.dropna(subset=['customer_id'])
print(f"[2] No-ID rows dropped : {before - len(customers_clean):,} rows")

# Step 3: Fill nulls using our strategy
for col in ['FN', 'Active']:
    if col in customers_clean.columns:
        customers_clean[col] = customers_clean[col].fillna(0).astype(int)

if 'club_member_status' in customers_clean.columns:
    customers_clean['club_member_status'] = customers_clean['club_member_status'].fillna('Unknown')

if 'fashion_news_frequency' in customers_clean.columns:
    customers_clean['fashion_news_frequency'] = customers_clean['fashion_news_frequency'].fillna('None')

if 'age' in customers_clean.columns:
    customers_clean['age'] = customers_clean['age'].fillna(customers_clean['age'].median())
    # Clip age to a reasonable range (sanity check)
    customers_clean['age'] = customers_clean['age'].clip(lower=15, upper=100)
    customers_clean['age'] = customers_clean['age'].astype(int)

# Step 4: Normalize customer_id to string (consistent format)
customers_clean['customer_id'] = customers_clean['customer_id'].astype(str).str.strip()

# Step 5: Reset index after all drops
customers_clean = customers_clean.reset_index(drop=True)

print(f"\n✅ Customers cleaning complete")
print(f"   Original rows : {len(customers_df):,}")
print(f"   Cleaned rows  : {len(customers_clean):,}")
print(f"   Remaining nulls: {customers_clean.isnull().sum().sum()}")

In [ ]:
# ── ARTICLES: Full Cleaning Experiment ───────────────────────────────────────
articles_clean = articles_df.copy()

# Step 1: Remove duplicate articles
before = len(articles_clean)
articles_clean = articles_clean.drop_duplicates(subset=['article_id'])
print(f"[1] Duplicates removed : {before - len(articles_clean):,} rows")

# Step 2: Drop rows with no article_id
before = len(articles_clean)
articles_clean = articles_clean.dropna(subset=['article_id'])
print(f"[2] No-ID rows dropped : {before - len(articles_clean):,} rows")

# Step 3: Fill nulls in all object columns
for col in articles_clean.select_dtypes(include='object').columns:
    null_count = articles_clean[col].isnull().sum()
    if null_count > 0:
        fill_val = 'No description' if 'desc' in col.lower() else 'Unknown'
        articles_clean[col] = articles_clean[col].fillna(fill_val)

# Step 4: Fill nulls in numeric columns
for col in articles_clean.select_dtypes(include=[np.number]).columns:
    if articles_clean[col].isnull().sum() > 0:
        articles_clean[col] = articles_clean[col].fillna(articles_clean[col].median())

# Step 5: Normalize article_id to string
articles_clean['article_id'] = articles_clean['article_id'].astype(str).str.strip()

# Step 6: Strip extra whitespace from text columns
for col in articles_clean.select_dtypes(include='object').columns:
    articles_clean[col] = articles_clean[col].astype(str).str.strip()

articles_clean = articles_clean.reset_index(drop=True)

print(f"\n✅ Articles cleaning complete")
print(f"   Original rows  : {len(articles_df):,}")
print(f"   Cleaned rows   : {len(articles_clean):,}")
print(f"   Remaining nulls: {articles_clean.isnull().sum().sum()}")

In [ ]:
# ── TRANSACTIONS: Full Cleaning Experiment ───────────────────────────────────
transactions_clean = transactions_df.copy()

# Step 1: Remove full-row duplicates (same purchase event recorded twice)
before = len(transactions_clean)
transactions_clean = transactions_clean.drop_duplicates()
print(f"[1] Full duplicates removed: {before - len(transactions_clean):,} rows")

# Step 2: Drop rows missing critical columns
before = len(transactions_clean)
transactions_clean = transactions_clean.dropna(subset=['customer_id', 'article_id', 't_dat'])
print(f"[2] Critical null rows dropped: {before - len(transactions_clean):,} rows")

# Step 3: Convert t_dat to datetime (critical for time-based features)
transactions_clean['t_dat'] = pd.to_datetime(transactions_clean['t_dat'])
print(f"[3] Date column converted to datetime")
print(f"    Date range: {transactions_clean['t_dat'].min()} → {transactions_clean['t_dat'].max()}")

# Step 4: Normalize ID types to string
transactions_clean['customer_id'] = transactions_clean['customer_id'].astype(str).str.strip()
transactions_clean['article_id']  = transactions_clean['article_id'].astype(str).str.strip()

# Step 5: Handle price column — fill with median if missing
if 'price' in transactions_clean.columns:
    null_price = transactions_clean['price'].isnull().sum()
    if null_price > 0:
        median_price = transactions_clean['price'].median()
        transactions_clean['price'] = transactions_clean['price'].fillna(median_price)
        print(f"[4] Price nulls filled: {null_price:,} rows")

# Step 6: Sort chronologically — essential for time-based train/test splitting
transactions_clean = transactions_clean.sort_values('t_dat').reset_index(drop=True)
print(f"[5] Sorted by date chronologically")

print(f"\n✅ Transactions cleaning complete")
print(f"   Original rows  : {len(transactions_df):,}")
print(f"   Cleaned rows   : {len(transactions_clean):,}")
print(f"   Remaining nulls: {transactions_clean.isnull().sum().sum()}")

## 🔗 Section 10 — Referential Integrity Check

In a recommendation system, data must be **linked correctly**:  
- Transactions reference customers → customers must exist in `customers.csv`  
- Transactions reference articles → articles must exist in `articles.csv`

Orphaned records (transactions pointing to non-existent customers/articles) cause silent failures in collaborative filtering.

In [18]:
# Check referential integrity after cleaning
all_customer_ids = set(customers_clean['customer_id'].unique())
all_article_ids  = set(articles_clean['article_id'].unique())

txn_customer_ids = set(transactions_clean['customer_id'].unique())
txn_article_ids  = set(transactions_clean['article_id'].unique())

# Customers in transactions but NOT in customers table
orphan_customers = txn_customer_ids - all_customer_ids
# Articles in transactions but NOT in articles table
orphan_articles  = txn_article_ids  - all_article_ids

print("=" * 55)
print("  REFERENTIAL INTEGRITY CHECK")
print("=" * 55)
print(f"  Unique customers in transactions : {len(txn_customer_ids):,}")
print(f"  Customers not in customers.csv   : {len(orphan_customers):,}")
print()
print(f"  Unique articles in transactions  : {len(txn_article_ids):,}")
print(f"  Articles not in articles.csv     : {len(orphan_articles):,}")
print()

if len(orphan_customers) == 0 and len(orphan_articles) == 0:
    print("  ✅ Referential integrity: PASSED — all IDs are consistent")
else:
    print("  ⚠️  Referential integrity: ISSUES FOUND — orphaned records detected")
    print("     → Consider filtering transactions to only known IDs")

  REFERENTIAL INTEGRITY CHECK
  Unique customers in transactions : 1,362,281
  Customers not in customers.csv   : 0

  Unique articles in transactions  : 104,547
  Articles not in articles.csv     : 0

  ✅ Referential integrity: PASSED — all IDs are consistent


## 💾 Section 11 — Save Temporary Cleaned Outputs

We save to `data/sample/` as **temporary experiment outputs**.  
These are NOT the production-ready files — those come from the `src/preprocessing/` scripts.  

Purpose of saving here:
- Quick validation that our logic works end-to-end
- Can be loaded into the next notebook (feature engineering) for early experimentation
- Easy to inspect output files manually

In [19]:
# Save cleaned versions as experiment outputs
customers_clean.to_csv(TEMP_PATH + 'customers_exp_clean.csv', index=False)
print(f"✅ customers_exp_clean.csv saved → {len(customers_clean):,} rows")

articles_clean.to_csv(TEMP_PATH + 'articles_exp_clean.csv', index=False)
print(f"✅ articles_exp_clean.csv   saved → {len(articles_clean):,} rows")

transactions_clean.to_csv(TEMP_PATH + 'transactions_exp_clean.csv', index=False)
print(f"✅ transactions_exp_clean.csv saved → {len(transactions_clean):,} rows")

print("\n📁 All experiment outputs saved to: data/sample/")

✅ customers_exp_clean.csv saved → 1,371,980 rows
✅ articles_exp_clean.csv   saved → 105,542 rows
✅ transactions_exp_clean.csv saved → 28,813,419 rows

📁 All experiment outputs saved to: data/sample/


## 📝 Section 12 — Summary & Next Steps

### ✅ What We Did
| Step | Result |
|------|--------|
| Missing value analysis | Identified and documented all null patterns |
| Duplicate analysis | Checked for row and key-based duplicates |
| Data type checks | Flagged mismatched types (strings, dates, IDs) |
| Date conversion | `t_dat` converted to datetime |
| Null handling | Filled with domain-appropriate values |
| Initial cleaning | All three datasets cleaned and verified |
| Referential integrity | Confirmed IDs are consistent across tables |

### 🔜 Next Steps
1. Move this logic to **`src/preprocessing/`** scripts (production-ready)
2. Run the production scripts to generate `data/processed/` files
3. Move to **`03_feature_engineering_experiments.ipynb`**
4. Build features: customer embeddings, article embeddings, interaction matrices

---

> **Engineering Note:** The preprocessing logic validated here is now ready to be  
> converted into the modular scripts in `src/preprocessing/`.  
> Those scripts can be run independently, tested with unit tests, and integrated  
> into an automated pipeline (e.g., Airflow, Prefect, or a simple Makefile).